# Description 

This notebook shows the results interpreter component. 

As before, the component comprises:

- A configuration 

- DomainData

- DomainTool 

+ a run function that takes a user query and returns either a str or a TaskResult 


# Initial target questions 

- query2_1 = "why 12 wells werent modelled from the initial screened ones?"
- query2_2 = "which is the best injector? in terms of utility"
- query2_3 = "whats the best supported producer?"
- query2_4 = "sumarize the quality of the models"
- query2_5 = "Is there any evidence of thieve zones?"
- query2_6 = "Plot the observed/simulated liquid rates for all the wells. " #analyst will take care of this. 
- query2_7 = "is there any evidence of aquifer support?"                    # fail 
- query2_8 = "Did we use BHP in the simulation?"                            # in the simulation?"                             
- query2_9 = "rank the injectors based on their utility"
- query2_10 = """for all producer wells, pick 3-4 points in their production history and for thos points plot the " \
observed liquid produced and simulated
"""
- query2_11 ="Give me a summary of the results, focus on support and chanelling"
- query2_12 = "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?"




# Dependencies


## External dependencies

In [ ]:
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')


import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict,  List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if


imported


## Internal dependencies 


In [2]:
from agentic_system.common.base_domain_tools import BaseDataComponent, BaseDomainTools
from agentic_system.common.get_llm import azure_llm_if as get_llm 


imported


# Initialize 

#### Mock data (taken from real cases)

- Model Logs
- Processing Logs  
- CRM-Liquid history match parameters table 
- CRM-Liquid history match rates table 
- Configuration file for the run
- Maybe injectors table 

for simplicity, we will use the same names as the files exported by the simulator 

Examples: 

- processing_logs.json, 

- historical_liquid_rates.csv 

- historical_liquid_crm.csv 

#### Initial target questions 

- query2_1 = "why 12 wells werent modelled from the initial screened ones?"
- query2_2 = "which is the best injector? in terms of utility"
- query2_3 = "whats the best supported producer?"
- query2_4 = "sumarize the quality of the models"
- query2_5 = "Is there any evidence of thieve zones?"
- query2_6 = "Plot the observed/simulated liquid rates for all the wells. "
- query2_7 = "is there any evidence of aquifer support?"                # fail 
- query2_8 = "Did we use BHP in the simulation?                                                    
- query2_9 = "rank the injectors based on their utility"
- query2_10 = """for all producer wells, pick 3-4 points in their production history and for thos points plot the observed liquid produced and simulated """
- query2_11 ="Give me a summary of the results, focus on support and chanelling"
- query2_12 = "Which producers appear poorly supported by injection, and how reliable is that conclusion given their history-match quality?"


# Onthologies  ? and subsurface responses ?  

CRM simulation background and objectives 

Injector utility 

Producer utility 

Producer support 

Stranded wells 

Connectivity 

Deficit and Aquifer support

Characteristic response time

High/Low injection rate, High/Low production 



? Reallocation
? Redistribution




##### We might need some pre-processing before making the data available 

In [3]:
import pandas as pd 
import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')



In [36]:

DATAFOLDER = "../../datasets/CRMResultsExample"
study_name = "ForAgents"
liquid_history_file = f"{DATAFOLDER}/historical_liquid_crm.csv"

liquid_crm = pd.read_csv( liquid_history_file )
if "ALLOCATION" in liquid_crm:
    liquid_crm.rename({"ALLOCATION":"GAIN"},axis=1,inplace=True)
liquid_crm['PALLOCATION'] = 0.75 *  liquid_crm['GAIN']
liquid_crm

,INJECTOR,PRODUCER,GAIN,TAU,TAUP,PRODUCTIVITY,LO,MODEL,ID,R2,BIAS_RATIO,CORRELATION,VARIANCE_RATIO,QUALITY_SCORE,SUBZONE,PALLOCATION
0,BG-1639_I,BG-2016_P,0.003472,10.293859,34.411707,0.0,1.194710,OneLayerCRMPSingleConstrained,0,-2.477959,0.961771,0.137599,1.659276,0.384530,ALLWARA,0.002604
1,BG-1799_I,BG-2016_P,0.049873,10.293859,34.411707,0.0,1.194710,OneLayerCRMPSingleConstrained,1,-2.477959,0.961771,0.137599,1.659276,0.384530,ALLWARA,0.037405
2,BG-1801_I,BG-2016_P,0.111089,10.293859,34.411707,0.0,1.194710,OneLayerCRMPSingleConstrained,2,-2.477959,0.961771,0.137599,1.659276,0.384530,ALLWARA,0.083317
3,BG-1957_I,BG-2016_P,0.000006,10.293859,34.411707,0.0,1.194710,OneLayerCRMPSingleConstrained,3,-2.477959,0.961771,0.137599,1.659276,0.384530,ALLWARA,0.000005
4,BG-1639_I,BG-2031_P,0.003376,48.684777,1.000000,0.0,1.000000,OneLayerCRMPSingleConstrained,4,0.293903,0.208798,0.997122,0.207998,0.413118,ALLWARA,0.002532
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1286,BG-2044_I,BG-1922_P,0.000029,0.500000,23.087441,0.0,0.048096,OneLayerCRMPSingleConstrained,1286,0.386908,0.999895,0.622034,0.617813,0.719121,ALLWARA,0.000022
1287,BG-2053_I,BG-1922_P,0.000091,0.500000,23.087441,0.0,0.048096,OneLayerCRMPSingleConstrained,1287,0.386908,0.999895,0.622034,0.617813,0.719121,ALLWARA,0.000068
1288,BG-2047_I,BG-2046_P,0.997503,0.525601,1.000000,0.0,1.000000,OneLayerCRMPSingleConstrained,1288,-4.850028,0.227819,0.466023,0.508528,0.351323,ALLWARA,0.748127
1289,BG-0740_I,BG-0760_P,0.000894,6.772576,23.732836,0.0,0.982152,OneLayerCRMPSingleConstrained,1289,0.932911,1.000937,0.965896,0.959762,0.973814,ALLWARA,0.000671


In [40]:
import pandas as pd

class CRMResultsPreprocess:

    NEGLIGIBLE_GAIN = 0.05
    VERY_LOW_GAIN = 0.10

    QUALITY_VERY_GOOD = 0.8
    QUALITY_GOOD = 0.6
    QUALITY_POOR = 0.4

    BHP_PRODUCTIVITY_TOLERANCE = 0.0001


    @classmethod
    def classify_gain(cls, gain: float) -> str:
        """
        Classify injector-producer connectivity strength.
        """

        if gain < cls.NEGLIGIBLE_GAIN:
            return "negligible"

        if gain < cls.VERY_LOW_GAIN:
            return "very_low"

        return "meaningful"


    @classmethod
    def classify_quality(cls, score: float) -> str:
        """
        Classify producer model quality from QUALITY_SCORE.
        """

        if score > cls.QUALITY_VERY_GOOD:
            return "very_good"

        if score >= cls.QUALITY_GOOD:
            return "good"

        if score >= cls.QUALITY_POOR:
            return "poor"

        return "very_poor"



    @classmethod
    def build_injector_summary(
        cls,
        connectivity: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Build one row per injector summarizing modeled connectivity.

        Injector utility is currently defined as:

            utility = sum(GAIN)

        Negligible connections (GAIN < NEGLIGIBLE_GAIN) contribute to the
        utility sum but do NOT count toward the number of supported producers.

        Returns
        -------
        pd.DataFrame
            Columns:
            - injector
            - utility
            - n_supported_producers
            - max_gain
            - dominant_producer
            - dominant_producer_quality_score
            - dominant_producer_quality_class
            - subzone, when available
        """

        required_columns = {
            "injector",
            "producer",
            "gain",
            "producer_quality_score",
            "producer_quality_class",
        }

        missing = required_columns - set(connectivity.columns)

        if missing:
            raise ValueError(
                f"Missing required connectivity columns: {sorted(missing)}"
            )

        rows = []

        for injector, group in connectivity.groupby("injector", sort=False):

            # Utility includes all fitted gains, including negligible ones.
            utility = group["gain"].sum()

            meaningful = group[
                group["gain"] >= cls.NEGLIGIBLE_GAIN
            ]

            n_supported_producers = meaningful["producer"].nunique()

            # Pair with the strongest modeled connectivity.
            dominant_idx = group["gain"].idxmax()
            dominant = group.loc[dominant_idx]

            row = {
                "injector": injector,
                "utility": utility,
                "n_supported_producers": n_supported_producers,
                "max_gain": dominant["gain"],
                "dominant_producer": dominant["producer"],
                "dominant_producer_quality_score":
                    dominant["producer_quality_score"],
                "dominant_producer_quality_class":
                    dominant["producer_quality_class"],
            }

            if "subzone" in group.columns:
                subzones = group["subzone"].dropna().unique()

                row["subzone"] = (
                    subzones[0]
                    if len(subzones) == 1
                    else ",".join(map(str, subzones))
                )

            rows.append(row)

        df = pd.DataFrame(rows)

        return (
            df
            .sort_values("utility", ascending=False)
            .reset_index(drop=True)
        )


    @classmethod
    def build_producer_support_summary(
        cls,
        connectivity: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Build one row per producer summarizing modeled injector support.

        Producer support is currently defined as:

            support = sum(GAIN)

        Negligible connections (GAIN < NEGLIGIBLE_GAIN) contribute to the
        support sum but do NOT count toward the number of supporting injectors.

        Returns
        -------
        pd.DataFrame
            Columns:
            - producer
            - support
            - n_supporting_injectors
            - max_gain
            - dominant_injector
            - quality_score
            - quality_class
            - subzone, when available
        """

        required_columns = {
            "injector",
            "producer",
            "gain",
            "producer_quality_score",
            "producer_quality_class",
        }

        missing = required_columns - set(connectivity.columns)

        if missing:
            raise ValueError(
                f"Missing required connectivity columns: {sorted(missing)}"
            )

        rows = []

        for producer, group in connectivity.groupby("producer", sort=False):

            # Support includes all fitted gains.
            support = group["gain"].sum()

            meaningful = group[
                group["gain"] >= cls.NEGLIGIBLE_GAIN
            ]

            n_supporting_injectors = meaningful["injector"].nunique()

            dominant_idx = group["gain"].idxmax()
            dominant = group.loc[dominant_idx]

            # Quality is producer-level and therefore should be constant
            # across every injector row for this producer.
            quality_scores = group["producer_quality_score"].unique()
            quality_classes = group["producer_quality_class"].unique()

            if len(quality_scores) != 1 or len(quality_classes) != 1:
                raise ValueError(
                    f"Inconsistent producer quality values for producer {producer}"
                )

            row = {
                "producer": producer,
                "support": support,
                "n_supporting_injectors": n_supporting_injectors,
                "max_gain": dominant["gain"],
                "dominant_injector": dominant["injector"],
                "quality_score": quality_scores[0],
                "quality_class": quality_classes[0],
            }

            if "subzone" in group.columns:
                subzones = group["subzone"].dropna().unique()

                row["subzone"] = (
                    subzones[0]
                    if len(subzones) == 1
                    else ",".join(map(str, subzones))
                )

            rows.append(row)

        df = pd.DataFrame(rows)

        return (
            df
            .sort_values("support", ascending=False)
            .reset_index(drop=True)
        )

    @classmethod
    def build_connectivity_table(
        cls,
        liquid_crm: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Build a normalized injector-producer connectivity table.

        Returns one row per modeled injector-producer pair.

        Semantic fields:
        - gain:
            Fraction of injector injection associated with liquid response
            at the producer.
        - pallocation:
            Fraction of the producer liquid production attributed to this injector.
        - gain_class:
            Connectivity strength classification based on GAIN.
        - producer_quality_score:
            History-match quality of the producer model.
        """

        required_columns = {
            "INJECTOR",
            "PRODUCER",
            "GAIN",
            "PALLOCATION",
            "QUALITY_SCORE",
        }

        missing = required_columns - set(liquid_crm.columns)

        if missing:
            raise ValueError(
                f"Missing required columns in liquid CRM results: {sorted(missing)}"
            )

        columns = [
            "INJECTOR",
            "PRODUCER",
            "GAIN",
            "PALLOCATION",
            "QUALITY_SCORE",
        ]

        if "SUBZONE" in liquid_crm.columns:
            columns.append("SUBZONE")

        df = liquid_crm[columns].copy()

        df = df.rename(
            columns={
                "INJECTOR": "injector",
                "PRODUCER": "producer",
                "GAIN": "gain",
                "PALLOCATION": "pallocation",
                "QUALITY_SCORE": "producer_quality_score",
                "SUBZONE": "subzone",
            }
        )

        numeric_columns = [
            "gain",
            "pallocation",
            "producer_quality_score",
        ]

        for column in numeric_columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

        if df[numeric_columns].isna().any().any():

            bad_columns = (
                df[numeric_columns]
                .columns[df[numeric_columns].isna().any()]
                .tolist()
            )

            raise ValueError(
                "Invalid or missing numeric values found in connectivity "
                f"columns: {bad_columns}"
            )

        df["gain_class"] = df["gain"].apply(
            cls.classify_gain
        )

        df["producer_quality_class"] = (
            df["producer_quality_score"]
            .apply(cls.classify_quality)
        )

        return df.reset_index(drop=True)


    @classmethod
    def build_producer_liquid_model_table(
        cls,
        liquid_crm: pd.DataFrame,
    ) -> pd.DataFrame:
        """
        Build a producer-level CRM liquid model table.

        Producer-level parameters and quality metrics are repeated in the raw
        table once per injector-producer pair. This method reduces them to one
        row per producer.

        bhp_used is simulation-level information. If any fitted producer has a
        non-negligible productivity value, BHP is considered to have been used
        by the simulation and the same value is assigned to every producer row.
        """

        required_columns = {
            "PRODUCER",
            "TAU",
            "TAUP",
            "PRODUCTIVITY",
            "LO",
            "BIAS_RATIO",
            "CORRELATION",
            "VARIANCE_RATIO",
            "QUALITY_SCORE",
        }

        missing = required_columns - set(liquid_crm.columns)

        if missing:
            raise ValueError(
                f"Missing required columns in liquid CRM results: {sorted(missing)}"
            )

        columns = [
            "PRODUCER",
            "TAU",
            "TAUP",
            "PRODUCTIVITY",
            "LO",
            "BIAS_RATIO",
            "CORRELATION",
            "VARIANCE_RATIO",
            "QUALITY_SCORE",
        ]

        if "SUBZONE" in liquid_crm.columns:
            columns.append("SUBZONE")

        df = liquid_crm[columns].copy()

        value_columns = [
            column
            for column in columns
            if column != "PRODUCER"
        ]

        inconsistent = (
            df.groupby("PRODUCER")[value_columns]
            .nunique(dropna=False)
            .gt(1)
            .any(axis=1)
        )

        bad_producers = (
            inconsistent[inconsistent]
            .index
            .tolist()
        )

        if bad_producers:
            raise ValueError(
                "Producer-level CRM values are not constant for producers: "
                f"{bad_producers}"
            )

        df = (
            df
            .drop_duplicates(subset=["PRODUCER"])
            .copy()
        )

        df = df.rename(
            columns={
                "PRODUCER": "producer",
                "TAU": "tau",
                "TAUP": "taup",
                "PRODUCTIVITY": "productivity",
                "LO": "lo",
                "BIAS_RATIO": "bias_ratio",
                "CORRELATION": "correlation",
                "VARIANCE_RATIO": "variance_ratio",
                "QUALITY_SCORE": "quality_score",
                "SUBZONE": "subzone",
            }
        )

        numeric_columns = [
            "tau",
            "taup",
            "productivity",
            "lo",
            "bias_ratio",
            "correlation",
            "variance_ratio",
            "quality_score",
        ]

        for column in numeric_columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

        if df[numeric_columns].isna().any().any():

            bad_columns = (
                df[numeric_columns]
                .columns[df[numeric_columns].isna().any()]
                .tolist()
            )

            raise ValueError(
                "Invalid or missing numeric values found in producer model "
                f"columns: {bad_columns}"
            )

        # BHP usage is simulation-level, not producer-level.
        bhp_used = bool(
            (df["productivity"] > cls.BHP_PRODUCTIVITY_TOLERANCE).any()
        )

        df["bhp_used"] = bhp_used

        df["quality_class"] = (
            df["quality_score"]
            .apply(cls.classify_quality)
        )

        return df.reset_index(drop=True)

    @classmethod
    def process_liquid_crm(
        cls,
        liquid_crm: pd.DataFrame,
    ) -> dict[str, pd.DataFrame]:

        connectivity = cls.build_connectivity_table(
            liquid_crm
        )

        producer_liquid_model = cls.build_producer_liquid_model_table(
            liquid_crm
        )

        injector_summary = cls.build_injector_summary(
            connectivity
        )

        producer_support_summary = cls.build_producer_support_summary(
            connectivity
        )

        return {
            "crm_connectivity": connectivity,
            "producer_liquid_model": producer_liquid_model,
            "injector_summary": injector_summary,
            "producer_support_summary": producer_support_summary,
        }

tables = CRMResultsPreprocess.process_liquid_crm( liquid_crm )
print(tables.keys())


tables['crm_connectivity'].head(3)

dict_keys(['crm_connectivity', 'producer_liquid_model', 'injector_summary', 'producer_support_summary'])


,injector,producer,gain,pallocation,producer_quality_score,subzone,gain_class,producer_quality_class
0,BG-1639_I,BG-2016_P,0.003472,0.002604,0.38453,ALLWARA,negligible,very_poor
1,BG-1799_I,BG-2016_P,0.049873,0.037405,0.38453,ALLWARA,negligible,very_poor
2,BG-1801_I,BG-2016_P,0.111089,0.083317,0.38453,ALLWARA,meaningful,very_poor


# First prototype 

In [1]:

import sys, pprint, pandas as pd , os, json, re, plotly.io as pio 
from pathlib import Path
from typing_extensions import Self
from typing import Any, cast
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

from agentic_system.common.base_domain_tools import BaseDataComponent, BaseDomainTools

class ResultsInterpreterData( BaseDataComponent ):

    #def __init__(self):
    #    super().__init__()
           
    
    def fetch_data_item_and_context( self, dataset_name: str )->str: 
        return 'dummy' 

    def record_response(self, question:str, response:str )->str:
        return 'dummy' 
    
    def interpret_data_item( self, question, dataset_name )->str:
        return  'dummy' 

    def _get_simulation_config( self ):
        """
        Retrieves the simulation configuration file as a JSON string and a description of 
        the parameters defined there. This file contains information about all the parameters used in the simulation.
        """
        pass 

    def _get_simulation_logs( self ):
        """
        Retrieves the simulation logs. These contain information on:
        - Errors in the modelling of specific wells
        - Reasons why specific wells were not modelled 
        - Other logs produced by the simulation engine 
        """
        pass 


    #def get_injector_performance_metrics():
    #    pass 

    #def get_injector_performance_metrics():
    #       pass 
       

    def used_bhp(self)->bool:
        return True 
    
class ResultsInterpreterTools(  BaseDomainTools[ResultsInterpreterData] ):

    #def __init__(self):
    #    super().__init__()


    @property
    def data(self) -> ResultsInterpreterData:
        return self.data_component

    @property
    def raw_data(self) -> Any:
        return self.data_component.raw_data 


    def compute_injector_summary(self)->pd.DataFrame:
        """
        Computes a summary of injector performance based on the CRM results.
        """
        raw_data = self.raw_data 

        return pd.DataFrame()  # Placeholder implementation

class ResultsInterpreterConfig( ):
    pass 

class ResultsInterpreterComponent( ):

    def __init__(self, llm, config:ResultsInterpreterConfig, data:ResultsInterpreterData, tools:ResultsInterpreterTools ):
        self.llm = llm 
        self.config = config 
        self.data_component = data 
        self.domain_tools = tools 
        self.domain_tools.set_data_component( self.data_component )

    def set_data(self, data:Any, metadata:Any|None = None)->Self:
        self.data_component.set_data(data, metadata)
        return self
    
    def run( self, query:str, background:str|None = None )->Any:
        message =  "[run] I am a hard-coded runner for CRMResultsAnalyst that creates an agent inplace"
        print(message)

        agent_tools = self.domain_tools.get_agent_tools()
        prompt = "You help with questions related to CRM results"

        agent = create_agent(model = self.llm,
                             system_prompt= prompt,
                             tools=agent_tools
                             )

      
        response = agent.invoke({"messages": [{"role": "user", "content": query}]})


        return response  

        # need to generate a basic plan 
    


Lets load some dummy data to work with. Here we load three tables: injectors, producers, locations

In [3]:
def get_config():
    return None 

# mock of DATAIKU setup  
class DataDrivenStorage:
        
    def __init__( self, config_vars ):
        pass 

    def get_project_dataset(self, project_name=None, filters=None):
        #path =  "../datasets/Demo1/"
        path =  Path("../../datasets/IX5I_4P/") 

        inj, prod, locs = self.fetch_data(path) 
        return inj, prod, locs

    def fetch_data(self,path:Path):
        inj  = pd.read_csv(path / "injectors.csv")
        pinj = pd.read_csv(path / "producers.csv")
        locs = pd.read_csv(path / "locations.csv")
        inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
        inj['DAY']   = inj['DATE'].dt.day
        inj['MONTH'] = inj['DATE'].dt.month
        inj['YEAR']  = inj['DATE'].dt.year
        pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
        pinj['DAY']   = pinj['DATE'].dt.day
        pinj['MONTH'] = pinj['DATE'].dt.month
        pinj['YEAR']  = pinj['DATE'].dt.year


        return inj, pinj, locs


In [4]:
# mock of fetching CRM input data 
inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
display( inj.head(2) )
display( prod.head(2))
display( locs.head(2))

# these tables are used by the visualization system (input data)

,DATE,NAME,WATER_INJECTION_VOLUME,SECTOR,ZONE,SUBZONE,WELL_TYPE,DAY,MONTH,YEAR
0,2015-12-02,I1,0.00,1,WARA,WARA1,Injector,2,12,2015
1,2016-01-02,I1,293.71,1,WARA,WARA1,Injector,2,1,2016


,DATE,NAME,LIQUID_VOLUME,SECTOR,ZONE,SUBZONE,WELL_TYPE,GAS_VOLUME,WATER_VOLUME,OIL_VOLUME,PRESSURE,DAY,MONTH,YEAR
0,2015-12-02,P1,0.000000,1,WARA,WARA1,Producer,0.000000,0.000000,0.000000,0,2,12,2015
1,2016-01-02,P1,910.803528,1,WARA,WARA1,Producer,240.452133,0.000001,910.803528,250,2,1,2016


,NAME,X,Y,SECTOR,ZONE,SUBZONE,WELL_TYPE
0,P1,1240,2040,1,WARA,WARA1,Producer
1,P2,440,1240,1,WARA,WARA1,Producer


In [5]:
# lets create smart data for those tables.
# For a fully operational SmartData object we need 
# 1. the data 
# 2. the semantic models 

# but we can initialize it from the semantic models and pass data later when we have it,
# we can initialize the object with both at the same time.
# every time data is "set" all previous tables are deleted.
# the semantic model doesnt change automatically. if needed, use the provided method. 



#This is a dictionary of table-name: semantic info
from agentic_system.common.known_tables_models import inj_prod_locs_semantic_catalog 
#inj_prod_locs_semantic_catalog
semantic_catalog = SemanticCatalog.model_validate(inj_prod_locs_semantic_catalog)
#type(semantic_catalog)

known_table_models = { t.name: t for t in semantic_catalog.tables } 
df_dict = {'injectors': inj, 'producers': prod , 'locations': locs }
#models = [ TableCard(x) for x in inj_prod_locs_semantic_catalog]

#one option 
smart_data = SmartData()
smart_data.init_from_semantic_models( semantic_catalog.tables )
smart_data.set_data( df_dict )



In [6]:
[x for x in dir(smart_data) if not x.startswith('_') ] 

['catalog_snapshot',
 'clear',
 'clear_derived',
 'conn',
 'execute_sql',
 'get_df',
 'get_single_table_brief_description',
 'get_table_as_df',
 'get_table_names',
 'get_tables_brief_description',
 'get_tables_creation_datetime',
 'init_from_data_and_models',
 'init_from_semantic_models',
 'initialize_from_named_dataframes',
 'register_derived_table',
 'restart_connection',
 'sanitize_df',
 'set_data']

In [7]:
print(smart_data.get_table_names())
print(smart_data.get_tables_brief_description())

smart_data.get_table_as_df('producers')
smart_data.get_single_table_brief_description('producers')

model = smart_data.catalog_snapshot('producers') #(), (['producers','locations'])

model.model_dump() 

['injectors', 'producers', 'locations']
{'injectors': 'Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', 'producers': 'Production time series. Each row contains a dated observation of produced volumes (oil,gas,water) and ratios (opional) for a given producer name, sector and subzone', 'locations': 'Data of well name, sector, well type and coordinates of the named well in each subzone.'}


{'base_tables': [{'name': 'producers',
   'description': 'Production time series. Each row contains a dated observation of produced volumes (oil,gas,water) and ratios (opional) for a given producer name, sector and subzone',
   'kind': 'base',
   'creation_date': '2026-08-26 13:54:56.019571',
   'row_count': 392,
   'columns': [{'name': 'DATE',
     'data_type': 'timestamp',
     'description': 'Production date.',
     'derived_column': False},
    {'name': 'NAME',
     'data_type': 'string',
     'description': 'Producer well identifier.',
     'derived_column': False},
    {'name': 'LIQUID_VOLUME',
     'data_type': 'float',
     'description': 'Total produced liquid.',
     'derived_column': False},
    {'name': 'WATER_VOLUME',
     'data_type': 'float',
     'description': 'Produced water.',
     'derived_column': False},
    {'name': 'GAS_VOLUME',
     'data_type': 'float',
     'description': 'Produced gas.',
     'derived_column': False},
    {'name': 'OIL_VOLUME',
     'data_ty

The key is that smart data can execute sql. 
We can pass the sql directly or get an agent to generate it.
If we use an agent, then we can pass to it the smart tools linked to smart data 
and it will be aware of the tables semantic models. 



In [8]:
result = smart_data.execute_sql(
    """
        SELECT
            NAME,
            SUM(WATER_INJECTION_VOLUME) AS total_injection
        FROM injectors
        GROUP BY NAME
        ORDER BY total_injection DESC
        LIMIT 3
    """)

print(result)


  NAME  total_injection
0   I1       157996.408
1   I2       125508.395
2   I5        96638.092


#### SmartTools

In [9]:
smart_tools = SmartDataTools( smart_data ) 

#these are tools for an agent:
tools = smart_tools.get_agent_tools() 

tools 

[StructuredTool(name='catalog_snapshot', description='Return an LLM-friendly textual snapshot of the data catalog.\n\nThis method is intended to ground agents with the available table\nschemas, descriptions, columns, and relevant metadata before they plan\nor execute data tasks.\n\nParameters\n----------\ninput_tables : None | str | Iterable[str], optional\n    Tables to include in the snapshot.\n\n    - None:\n        Include all tables in the catalog.\n    - str:\n        Include only the table with this name.\n    - Iterable[str]:\n        Include only the listed table names.\n\nReturns\n-------\nstr\n    A structured, readable catalog description suitable for use in\n    planner prompts, executor prompts, and schema-grounded reasoning.', args_schema=<class 'langchain_core.utils.pydantic.catalog_snapshot'>, func=<bound method SmartDataTools.catalog_snapshot of <agentic_system.visualization.smart_data_tools.SmartDataTools object at 0x0000015F06809150>>),
 StructuredTool(name='get_sin

#### An agent to query these tables via natural language 

 - Embelishments 

Before moving to an agent, we need a couple of embelishments.
These are to secure that the SQL code is compliant with the sql-engine running behind the scenes. Also to secure that it is syntactically correct. These embelishments are:

- Constrains
- SQL flavour specific. Here we will use duckdb flavour.

These will enter in the agent prompt later as:

- {idiom}
- {idiom_examples}
- {constraints}



In [10]:
from agentic_system.common.known_idioms import idioms as all_idioms

idiom = 'duckdb'
idiom_rules = all_idioms[ idiom ]

constraints = "".join([f"- {i}\n" for i in semantic_catalog.semantic_constraints])
idiom_context = "".join([f"- {i}: {v}\n" for i,v in idiom_rules.items()])



In [11]:
anayst_prompt_template1d = """
You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database


===============================================================================
Workflow:
===============================================================================
You must:
1. Always call catalog_snapshot as the first step of the workflow.
After catalog_snapshot, check whether all requested concepts map clearly to catalog tables, columns, or known metrics.
If not, ask clarification and do not call sql_* tools.

2. Analyze the question and the information in the catalog and produce a concise PLAN
The PLAN must be concise and must include:
- required source tables
- whether existing derived tables can be reused
- target table names to materialize
- high-level transformation logic, without SQL


3. You MUST ALWAYS record the PLAN in plain text. Only after the PLAN message is sent may you call sql_* tools.
4. Use sql_materialize to create intermediate tables.
5. When multiple output tables are to be produced, proceed sequentially one at a time  
6. Your job finishes once all the target tables are confirmed present (either via initial audit or your materializations).  

===============================================================================
Important:
===============================================================================
- The name of generated tables and columns should reflect the table contents  

- Use lowercase snake_case for table names and column names 
    Example 1: yearly_aggregated_oil_producer_per_subzone
    Example 2: gas_oil_water_cummulated_volumes 

- Be explicit in the detailed description of tables produced 

- Sequential Execution:  If you need to materialize multiple tables, do so one by one, verifying the metadata for each.

===============================================================================
Output
===============================================================================
In each turn you will provide as result one or more tables 

Do not proceed if the question cannot be answered with the available data. 
Instead, ask for clarification 

===============================================================================
SQL generation rules 
===============================================================================
- ALWAYS use **{idiom}** compliant SQL syntax when generating queries.
{idiom_examples}

===============================================================================
Domain constraints
===============================================================================
{constraints}

===============================================================================
Chart-ready output rules
===============================================================================
CHART-READY OUTPUT RULES

For chart/plot/graph requests:

- "plot A by B"
  => return one row per B

- "plot A by B,C"
  => return one row per (B,C)

- "plot A by B,C,D"
  => return one row per (B,C,D)

Rules:
- Preserve all grouping columns.
- Aggregate A at the requested grouping level.
- Use sum by default for additive quantities unless another aggregation is requested.
- If multiple grouping columns together naturally define the chart axis, also create a readable display label column.
- Do not return raw detail rows for grouped chart requests.

Do not return raw detail rows when the user asks for aggregated chart-ready output.

===============================================================================
MANDATORY REUSE RULE:
===============================================================================

After calling catalog_snapshot:

1. If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.

2. You MUST NOT recompute intermediate tables if an equivalent derived table already exists.

3. Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.


Important:
- If a query depends on a table, ensure it has been materialized first.
- Always ask for clarification if the question is ambiguous or cannot be answered with the available data.



"""


prompt = anayst_prompt_template1d.format(idiom = idiom, 
                                    idiom_examples = idiom_context, 
                                    constraints = constraints)


Finally the agent 

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage
from typing import Any, Dict, Generic, List, Iterable, Literal, TypeVar, Union, Optional,TypedDict
from typing_extensions import Self
from agentic_system.common.get_llm import azure_llm_if
from pathlib import Path
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent


In [13]:

from agentic_system.common.structured_responses import AgentTableResponse

llm = get_llm() 
query = "which are the two injectors with the highest cummulated water injection "

agent = create_agent(
            model=llm,
            system_prompt = prompt,
            tools=tools,
            #response_format=ToolStrategy(AgentTableResponse),
        )

response = agent.invoke({
            "messages": [
                {"role": "user", "content": query}
            ]
        })

raw_result = response.get("structured_response")


zero temp, seed 42, top_p = 1


In [14]:
response['messages'][-1]

AIMessage(content='The two injectors with the highest cumulative water injection volumes are:\n\n1. Injector `I1` with a total water injection volume of **157,996.41**.\n2. Injector `I2` with a total water injection volume of **125,508.40**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 2965, 'total_tokens': 3024, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 2688}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 43, 'engine_ttlt_ms': 363, 'pre_inference_ms': 624, 'service_tbt_ms': 6, 'service_ttft_ms': 748, 'service_ttlt_ms': 1064, 'total_duration_ms': 452, 'user_visible_ttft_ms': 124}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_85fa909971', 'id': 'chatcmpl-EH7RwRBJTvw8jzIilkkQUpRhKi8vq', 'service_tier': 

In [15]:
raw_result

In [16]:
#print( response )
#table_name = "top_two_injectors_by_water_injection"
table_name = raw_result.tables[0].table_name 


smart_data.get_table_as_df( table_name )

AttributeError: 'NoneType' object has no attribute 'tables'

# The DataAnalystComponent 
Basically encloses in one class all the previous functionality.
You need to configure it (semantic, idioms) and call "run" on it. 
It should take care of everything without intervention.


In [17]:

from dataclasses import dataclass

from agentic_system.common.base_domain_tools import BaseDomainTools
from agentic_system.common.structured_responses import TaskResult
from agentic_system.visualization.smart_data import SmartData
from agentic_system.visualization.smart_data_tools import SmartDataTools
from agentic_system.common.structured_responses import *
from agentic_system.visualization.prompts import anayst_prompt_template

@dataclass 
class DataAnalystConfig:
    
    prompt_template :str# = 'anayst_prompt_template' #depends on idioms, constraints, etc.
    prompt: str | None = None     
    use_structured_output : bool = True 

    def build_prompt( self, few_shot_examples = None , background = None ):
        '''
        builds the data_analyst prompt using all the semantics + 
        few_shot examples if any and background info if any
        '''
        pass 


class DataAnalystComponent:
    agent_name = "data_analysis"

    def __init__(
            self,
            llm: Any,
            config: DataAnalystConfig, # | None = None,
            smart_data: SmartData | None = None,
            domain_tools: BaseDomainTools | None = None 
        ):
        self.llm = llm
        self._smart_data = smart_data if smart_data is not None else SmartData()
        self.config = config #if config #is not None else SQLAnalystConfig()

        self.tools_object = SmartDataTools(self._smart_data)
     
        self.domain_tools = domain_tools 
        self.tools = self.tools_object.get_tools()

        if domain_tools:
            self.tools = self.tools + self.domain_tools.get_agent_tools() # type: ignore

    @property
    def prompt(self) -> str:
        return self.config.prompt # type: ignore

    @property
    def smart_data(self) -> SmartData:
        return self._smart_data
    
    def init_semantic_models(self,semantic_catalog, idiom_rules, idiom = 'duckdb'): 
        
        
        # the tables
        #first set the semantic model, when data arrives later we set the new data. No tools or anything will need update
        #this has no data, just semantic models and we dont know the size of the tables 
        self._smart_data.init_from_semantic_models( semantic_catalog.tables )

        # the base prompt (without history, which could be added)
        constraints = "".join([f"- {i}\n" for i in semantic_catalog.semantic_constraints])
        idiom_context = "".join([f"- {i}: {v}\n" for i,v in idiom_rules.items()])
        self.config.prompt = self.config.prompt_template.format(idiom = idiom, 
                                    idiom_examples = idiom_context, 
                                    constraints = constraints)

    def set_data(self,df_dict: Dict[str,pd.DataFrame]):
         self._smart_data.set_data(df_dict) 

    def run(self, query: str, facts_context : str | None  = None) -> TaskResult | str :

        prompt = self.prompt
        if  facts_context:
            prompt = prompt + f"\nCONVERSATION FACTS:\n{facts_context}" 


        response_format = ToolStrategy(AgentTableResponse) if self.config.use_structured_output else None 

        agent = create_agent(
            model=self.llm,
            system_prompt = prompt,
            tools=self.tools,
            response_format=response_format,
        )

        response = agent.invoke({
            "messages": [
                {"role": "user", "content": query}
            ]
        })

        if not response_format:
            #return response 
            return response['messages'][-1].content 


        raw_result = response.get("structured_response")

        if raw_result is None:
            raw_results = []
        elif isinstance(raw_result, list):
            raw_results = raw_result
        else:
            raw_results = [raw_result]

        cheap_parts: list[str] = []
        data_results: list[DataFrameResult] = []

        for raw in raw_results:
            text = getattr(raw, "text", None)
            if text:
                cheap_parts.append(text)

            tables = getattr(raw, "tables", []) or []

            for table in tables:
                table_name = getattr(table, "table_name", None)
                description = getattr(table, "description", None)

                if not table_name:
                    continue

                cheap_parts.append(
                    f"{self.agent_name} agent created and stored the table `{table_name}`: {description or 'No description provided.'}"
                )

                df = self._smart_data.get_table_as_df(table_name)

                data_results.append(
                    DataFrameResult(
                        table_name=table_name,
                        description=description,
                        dataframe=df,
                    )
                )

                if df.shape[0] < 10 and df.shape[1] < 4:
                    cheap_parts.append(
                        f"Table `{table_name}` contents:\n{df.to_string(index=False)}"
                    )

        cheap_output = "\n\n".join(cheap_parts) if cheap_parts else (
            "Data analyst completed, but no textual summary or table metadata was returned."
        )

        return TaskResult(
            agent=self.agent_name,
            instruction=query,
            cheap_output=cheap_output,
            raw_results=raw_results,
            data_results=data_results,
        )


analyst = DataAnalystComponent(llm=llm, 
                               config=DataAnalystConfig(prompt_template=anayst_prompt_template, 
                                                        use_structured_output=True))


analyst.init_semantic_models(semantic_catalog, idiom_rules, idiom = 'duckdb')
analyst.set_data( df_dict )


        

In [18]:
query = "which are the two injectors with the highest cummulated water injection "

response = analyst.run( query )


===== PLAN (TOOL) =====
1. Source table: injectors
2. Aggregate WATER_INJECTION_VOLUME by NAME to calculate the cumulative water injection for each injector.
3. Sort the results in descending order of cumulative water injection.
4. Select the top two injectors with the highest cumulative water injection.
5. Materialize the result as a table named top_two_injectors_by_water_injection.



In [19]:
response

TaskResult(agent='data_analysis', instruction='which are the two injectors with the highest cummulated water injection ', cheap_output='data_analysis agent created and stored the table `top_two_injectors_by_water_injection`: This table contains the top two injectors with the highest cumulative water injection volumes. It includes the injector well identifier (NAME) and the total water injection volume (total_water_injection).\n\nTable `top_two_injectors_by_water_injection` contents:\nNAME  total_water_injection\n  I1             157996.408\n  I2             125508.395', raw_results=[AgentTableResponse(agent='analyst', clarification=None, user_query='which are the two injectors with the highest cummulated water injection', tables=[TableItemAgentResponse(table_name='top_two_injectors_by_water_injection', description='This table contains the top two injectors with the highest cumulative water injection volumes. It includes the injector well identifier (NAME) and the total water injection 

# End 

In [ ]:

def init_visualization_system( llm ):
   

    vis_system = VisualizationAgenticSystem( llm )


    idiom = 'duckdb'
    idiom_rules = all_idiom_rules[idiom]
    semantic_catalog_model = SemanticCatalog.model_validate( semantic_catalog )

    analyst = vis_system.data_analyst_component
    analyst.init_semantic_models( semantic_catalog_model,idiom_rules)
    



    return vis_system


llm = azure_llm_if()
vis_system = init_visualization_system(llm)


# data changes
# this mocks data comming from the UI
# so we just update tge analyst 
inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
vis_system.data_analyst_component.set_data( {'injectors':inj, 
                                             'producers':prod, 
                                             'locations': locs } )



In [ ]:
#from visualization_system.visualization_backend.analyst.analyst_backend import VisualizationAgenticSystem


In [ ]:

# these are just mocks 


from visualization_system.visualization_backend.analyst.analyst_backend import VisualizationAgenticSystem
 
def get_config():
    return None 

class DataDrivenStorage:
        
    def __init__( self, config_vars ):
        pass 

    def get_project_dataset(self, project_name=None, filters=None):
        #path =  "../datasets/Demo1/"
        path =  Path("../datasets/IX5I_4P/") 

        inj, prod, locs = self.fetch_data(path) 
        return inj, prod, locs

    def fetch_data(self,path:Path):
        inj  = pd.read_csv(path / "injectors.csv")
        pinj = pd.read_csv(path / "producers.csv")
        locs = pd.read_csv(path / "locations.csv")
        inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
        inj['DAY']   = inj['DATE'].dt.day
        inj['MONTH'] = inj['DATE'].dt.month
        inj['YEAR']  = inj['DATE'].dt.year
        pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
        pinj['DAY']   = pinj['DATE'].dt.day
        pinj['MONTH'] = pinj['DATE'].dt.month
        pinj['YEAR']  = pinj['DATE'].dt.year


        return inj, pinj, locs

def init_visualization_system( llm ):
   

    vis_system = VisualizationAgenticSystem( llm )


    idiom = 'duckdb'
    idiom_rules = all_idiom_rules[idiom]
    semantic_catalog_model = SemanticCatalog.model_validate( semantic_catalog )

    analyst = vis_system.data_analyst_component
    analyst.init_semantic_models( semantic_catalog_model,idiom_rules)
    



    return vis_system


llm = azure_llm_if()
vis_system = init_visualization_system(llm)


# data changes
# this mocks data comming from the UI
# so we just update tge analyst 
inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
vis_system.data_analyst_component.set_data( {'injectors':inj, 
                                             'producers':prod, 
                                             'locations': locs } )





In [ ]:

query2 = """
List the 5 wells with the highest water cut in current date
"""



#this is what the presenter consumes 
execution_state = vis_system.run( query2 )


In [ ]:

from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
print(ui_items)

for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 




In [ ]:
query2 = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production since year 2015 for all the wells 
"""

planner = vis_system.planner_component
plan = planner.run( query )

In [ ]:
pprint.pprint( plan.model_dump())


In [ ]:
direct_answer = vis_system.direct_answer_component

direct_answer.run(  plan.tasks[0].instruction  ).data_results

In [ ]:

analyst = vis_system.data_analyst_component
task_results = [ analyst.run(task.instruction) for task in plan.tasks[1:] ]



In [ ]:
task_results

In [ ]:
from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = [ presenter.process_single_task_result(result)[0] for result in task_results ] 





In [ ]:
ui_items

In [ ]:


for item in ui_items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 


In [ ]:
query = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production for each year since year 2015 for all the wells 
"""

state = vis_system.run( query )


In [ ]:
presenter = PresenterComponent4( llm )

ui_items = presenter.run(state)

In [ ]:
ui_items.items

In [ ]:
for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 

In [ ]:
state 

In [ ]:
from visualization_system.visualization_backend.analyst.analyst_backend import DataFrameResult, ExecutorState, PresenterConfig, PresenterResponse, SubInstructions, TableResponseProcessor, TaskResult, TextResult, UIItem
  


In [ ]:
from dataclasses import dataclass
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')
import warnings

from typing import Any, Dict, Generic, List, Iterable, Literal, TypeVar, Union, Optional,TypedDict
from typing_extensions import Self
   
from uuid import uuid4
 
from pydantic import BaseModel, Field 
from visualization_system.visualization_backend.get_llm_model import azure_llm_if
from pathlib import Path
from langchain.agents.structured_output import ToolStrategy
from langgraph.graph import StateGraph, END

from langchain.agents import create_agent

from visualization_system.common.base_plan import PlannerConfig
#from visualization_system.visualization_backend.analyst.analyst_system import PlannerConfig, DirectAnswerConfig
from visualization_system.visualization_backend.analyst.analyst_models import TableItemAgentResponse, VisualizationSystemPlanner, VisualizationSystemTask, VisualizationSystemPlan 
from visualization_system.visualization_backend.analyst.smart_data import SmartData
from visualization_system.visualization_backend.analyst.smart_data_tools import SmartDataTools
from visualization_system.visualization_backend.analyst.analyst_system import SQLAnalystConfig

from visualization_system.visualization_backend.analyst.prompts import visualization_planner_prompt3 
from visualization_system.visualization_backend.analyst.prompts import anayst_prompt_template 
from visualization_system.visualization_backend.analyst.prompts import chart_agent_prompt
from visualization_system.visualization_backend.analyst.prompts import small_table_prompt
from visualization_system.visualization_backend.analyst.prompts import split_subinstructions_prompt

 


import pandas as pd, re, json  
from langchain_core.messages import SystemMessage, HumanMessage
from visualization_system.visualization_backend.global_models import UIState



In [ ]:
from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4

CHART_AGENT_PROMPTV4 = """
You are a chart planning agent.

You receive:
- user query
- table summaries
- column names, roles, cardinality, and descriptions

Return a JSON plan with:
- zero or more ordered preprocess operations 
- exactly ONE plot step. 
- ONE plot step (see <plot_tools> below) must be in the plan regardless of whether there are or not preprocess operations

 
PREPROCESSING

The preprocess field is an ordered list of operations applied before plotting.

Supported operations:

1. create_combined_category
   Creates a category column from two existing columns.

Args:
{
"operation": "create_combined_category",
"args": {
"col1": "<column>",
"col2": "<column>",
"new_col": "<new_column>",
"sep": " / "
}
}

2. create_date_bucket
   Creates a date grouping column.

Args:
{
"operation": "create_date_bucket",
"args": {
"date_col": "<date_column>",
"bucket": "D|W|M|Q|Y",
"new_col": "<new_column>"
}
}

3. filter_rows
   Keeps rows matching one or more conditions.

Args:
{
"operation": "filter_rows",
"args": {
"filters": [
{
"column": "<column>",
"operator": "==|!=|>|>=|<|<=|in|not_in",
"value": "<value_or_list>"
}
]
}
}

4. aggregate
   Groups and aggregates the data before plotting.

Args:
{
"operation": "aggregate",
"args": {
"group_by": ["<column>", "..."],
"metrics": {
"<numeric_column>": "sum|mean|median|min|max|count|nunique"
}
}
}

5. sort_rows
   Sorts the rows.

Args:
{
"operation": "sort_rows",
"args": {
"sort_by": "<column_or_list>",
"ascending": true|false
}
}

6. limit_rows
   Keeps only the first N rows.

Args:
{
"operation": "limit_rows",
"args": {
"n": <integer>
}
}

7. select_columns
   Keeps only selected columns.

Args:
{
"operation": "select_columns",
"args": {
"columns": ["<column>", "..."]
}
}

8. select_top_entities
   Selects the top or bottom entities using a metric.

Use keep_all_rows = true when the ranking period is only used to identify entities, but the final chart needs all rows for those entities.

Args:
{
"operation": "select_top_entities",
"args": {
"entity_col": "<entity_column>",
"metric_col": "<numeric_column>",
"n": <integer>,
"aggregate": "sum|mean|median|min|max|count|nunique",
"ascending": true|false,
"filters": [],
"keep_all_rows": true|false
}
}

Rules:

* Use preprocess only when the input table is not already ready for plotting.
* Operations are executed in the listed order.
* Do not invent columns.
* Prefer the smallest number of operations needed.
* Aggregation, filtering, ranking, date bucketing and limiting should be done in preprocess rather than in the plotting tool.



PLOT TOOLS



Args:
{
  "columns": ["<column>", "..."],
  "sort_by": null | "<column>",
  "sort_order": "asc|desc",
  "limit": null | <integer>,
  "title": "<title>"
}


plot_bar_chart:
Use for comparing one or more quantitative values across categorical or bucketed temporal groups.

Best for:
- "Y by A"
- "Y per A"
- "Y by A and B"
- totals, averages, counts, rankings, grouped comparisons

Mapping rules:
- For "Y by A": use x = A, y = Y, group_by = [A].
- For "Y by A and B": use x = A, color_by = B, y = Y, group_by = [A, B].
- For "Y by A, B, and C": use x = A, color_by = B or C, and group_by = [A, B, C].
- If two columns together define the x-axis label, create the combined column first with preprocess_for_chart and use it as x.
- group_by must include every column needed to preserve the requested breakdown.
- Use aggregate = "sum" by default for additive quantities unless the query specifies another aggregation.

Do not use a bar chart for multi-period time-series trends when a line chart can show the evolution more clearly.

A temporal column does not automatically make a bar chart appropriate.
Use bars for discrete period totals only when the user explicitly asks to compare independent periods or requests a bar chart.

Args:
{
  "x": "<category_or_bucket_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>"],
  "color_by": null | "<secondary_category_col>",
  "orientation": "v|h",
  "barmode": "group|stack|relative",
  "title": "<title>"
}

plot_line_chart:
Use for trends, time series, ordered progression, or cumulative values over time.

Use a line chart when:
- x is a date, year, month, quarter, or another ordered temporal column;
- the user asks for yearly, monthly, quarterly, or daily evolution;
- the chart shows how a metric changes across multiple time periods;
- multiple entities should be represented as separate time-series traces.

For "Y by time for each A":
- x = time column
- y = Y
- series_by = A
- each unique series_by value becomes one trace

Prefer a line chart over a bar chart whenever the main purpose is to show change or evolution over time.

Trace rules:
- Use series_by when one column defines separate traces.
- series_by values become the trace names.
- Use series_by = NAME when each well should be a separate trace.
- If y is a list and series_by is provided, traces are named "<series_by value> - <y column>".
- If series_by is null, traces are named from y column names.
- color_by is deprecated. Use series_by instead.

Args:
{
  "x": "<time_or_ordered_col>",
  "y": "<numeric_col_or_list>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<group_col>", "..."],
  "series_by": null | "<category_col>",
  "date_bucket": null | "D|W|M|Q|Y",
  "cumulative": true|false,
  "title": "<title>"
}

plot_pie_chart:
Use only for part-to-whole/share/composition questions.
Args:
{
  "labels": "<category_col>",
  "values": "<numeric_col>",
  "aggregate": null | "sum|mean|median|min|max|count|nunique",
  "group_by": null | ["<label_col>"],
  "hole": 0.0,
  "title": "<title>"
}

plot_scatter_chart:
Use for numeric-vs-numeric relationships, correlations, crossplots, clusters, or row-level comparisons.
Args:
{
  "x": "<numeric_col>",
  "y": "<numeric_col_or_list>",
  "series_by": null | "<category_col>",
  "size_by": null | "<numeric_col>",
  "text_by": null | "<label_col>",
  "title": "<title>"
}

IMPORTANT
YOU MUST address only the parts of the user question for which the table is related
YOU MUST Ignore the parts of the question that the information in the table cannot address
             

RULES
- Return only valid JSON.
- Do not invent tools.
- Do not invent arguments.
- Use only columns that exist or are created by preprocess_for_chart.
- Prefer no preprocess when existing columns are sufficient.
- Use sum by default for additive quantities unless otherwise specified.
- If uncertain, return {"reason": "...", "preprocess": null, "plot": null}.
- When multiple temporal dimensions together define the displayed x-axis grouping
(e.g. year + quarter, year + month),
create a combined temporal category for x.

OUTPUT SHAPE
{

  "preprocess": [],
  "plot": {
    "tool": "<plot_tool>",
    "args": {}
  }
}
"""

chart_agent_prompt2 = CHART_AGENT_PROMPTV4

c = PresenterConfig( )
c.prompt = chart_agent_prompt 

In [ ]:
query = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production since year 2015 for all the wells 
"""

planner = vis_system.planner_component
plan = planner.run( query )


In [ ]:


query2 = """Explain VRR briefly and then:

1. plot the yearly liquid production of the 5 top producers based on the cummulated oil production in 2018, 
2. show the cummulated liquid production since year 2015 for the first two of those wells.  

"""


#this is what the presenter consumes 
execution_state = vis_system.run( query2 )



In [ ]:
import pickle
with open("execution_state.pkl", "wb") as file:
    pickle.dump(execution_state, file)


In [ ]:
import pickle 
with open("execution_state.pkl", "rb") as file:
    loaded_data = pickle.load(file)

#import pickle
#with open("execution_state.pkl", "wb") as file:
#    pickle.dump(execution_state, file)

execution_state = loaded_data

In [ ]:
execution_state

In [ ]:

from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
print(ui_items)

for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 




In [ ]:

from visualization_system.visualization_backend.analyst.analyst_backend import PresenterComponent4


presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
ui_items

In [ ]:
ui_items.items

In [ ]:
for item in ui_items.items:

    try:
        if item.type == "text":
            print( item.data['text'])
        if item.type == "chart":
            item = item.data['plotly']
            pio.show(item)
    except Exception as e:
        print(e)
        pass 
        

# END 

In [ ]:
#item = ui_items.items[1]
#item = item.data['plotly']
#pio.show(item)

item = ui_items.items[3]
item = item.data['plotly']
pio.show(item)




# Improved the presenter. 

In [ ]:
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if

from visualization_system.visualization_backend.all_classes import * 



In [ ]:
llm = azure_llm_if()
import pickle 
with open("execution_state.pkl", "rb") as file:
    loaded_data = pickle.load(file)

#import pickle
#with open("execution_state.pkl", "wb") as file:
#    pickle.dump(execution_state, file)

execution_state = loaded_data


presenter = PresenterComponent4(llm)

presenter_response = presenter.run(execution_state)

ui_items = presenter_response.items

ui_items 

In [ ]:
for item in ui_items:
    if item.type=='text':
        print( item.data['text'])
    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


# END 

In [ ]:
from visualization_system.visualization_backend.analyst.prompts import chart_agent_prompt
from visualization_system.visualization_backend.analyst.prompts import small_table_prompt
from visualization_system.visualization_backend.analyst.prompts import split_subinstructions_prompt

class SubInstruction(BaseModel):
    """
    One presentation item to produce from one source.
    """

    sub_instruction: str = Field(
        description=(
            "The specific part of the original instruction that this source "
            "should answer."
        )
    )

    kind: Literal["table", "text"] = Field(
        description="Whether the source is a table or a text result."
    )

    source_id: str = Field(
        description=(
            "The exact SOURCE_ID provided in the available sources. "
            "For tables use the table name. "
            "For text use the text SOURCE_ID."
        )
    )


class SubInstructions(BaseModel):
    """
    Ordered presentation plan for one TaskResult.
    """

    items: list[SubInstruction] = Field(
        description=(
            "The ordered list of presentation items to generate."
        )
    )
    
    
class PresenterChartingTools:

    ALLOWED_AGGS = {"sum", "mean", "median", "min", "max", "count", "nunique"}


    plotly_config = {
                "responsive": True,
                "displaylogo": False,
            }



    def run_preprocess(
        self,
        df: pd.DataFrame,
        preprocess_steps: list[dict] | None,
    ) -> pd.DataFrame:
        work = df.copy()

        for step in preprocess_steps or []:
            
            operation = step.get("operation")

            if not operation:
                raise ValueError("Preprocess step is missing 'operation'")

            args = step.get("args") or {}

            work = self.run_preprocess_operation(
                operation=operation,
                df=work,
                args=args,
            )

        return work


    def run_preprocess_operation(
        self,
        operation: str,
        df: pd.DataFrame,
        args: dict[str, Any],
    ) -> pd.DataFrame:
        preprocess_tools = {
            "filter_rows": self.filter_rows,
            "aggregate": self.aggregate_for_chart,
            "sort_rows": self.sort_rows,
            "limit_rows": self.limit_rows,
            "select_columns": self.select_columns,
            "select_top_entities": self.select_top_entities,
            "create_combined_category": self.create_combined_category,
            "create_date_bucket": self.create_date_bucket,
        }

        if operation not in preprocess_tools:
            raise ValueError(f"Unknown preprocess operation: {operation}")

        return preprocess_tools[operation](
            df=df,
            **args,
        )


    ##########################
    #       pre-process      # 
    ##########################
    def filter_rows(self,df: pd.DataFrame,filters: list[dict]) -> pd.DataFrame:
        
        work = df.copy()
        for item in filters:
            column = item["column"]
            operator = item["operator"]
            value = item["value"]

            self._validate_columns(work, [column])

            if operator == "==":
                work = work[work[column] == value]
            elif operator == "!=":
                work = work[work[column] != value]
            elif operator == ">":
                work = work[work[column] > value]
            elif operator == ">=":
                work = work[work[column] >= value]
            elif operator == "<":
                work = work[work[column] < value]
            elif operator == "<=":
                work = work[work[column] <= value]
 


            elif operator == "in":
                if not isinstance(value, (list, tuple, set)):
                    raise ValueError("'in' filter value must be a list")
                work = work[work[column].isin(value)]

            elif operator == "not_in":
                if not isinstance(value, (list, tuple, set)):
                    raise ValueError("'not_in' filter value must be a list")
                work = work[~work[column].isin(value)]




            else:
                raise ValueError(f"Unsupported filter operator: {operator}")

        return work

    def aggregate_for_chart( self, df: pd.DataFrame, group_by: list[str],
        metrics: dict[str, str],
    ) -> pd.DataFrame:
        
        self._validate_columns(df, group_by)

        for column, aggregate in metrics.items():
            self._validate_columns(df, [column])

            if aggregate not in self.ALLOWED_AGGS:
                raise ValueError(f"Unsupported aggregate: {aggregate}")

        return (df.groupby(group_by, dropna=False, as_index=False).agg(metrics))

    def sort_rows(
        self,
        df: pd.DataFrame,
        sort_by: str | list[str],
        ascending: bool = True,
    ) -> pd.DataFrame:
        sort_columns = self._as_list(sort_by)
        self._validate_columns(df, sort_columns)

        return df.sort_values(
            sort_columns,
            ascending=ascending,
        )

    def limit_rows(
        self,
        df: pd.DataFrame,
        n: int,
    ) -> pd.DataFrame:
        return df.head(n)

    def select_columns(
        self,
        df: pd.DataFrame,
        columns: list[str],
    ) -> pd.DataFrame:
        self._validate_columns(df, columns)
        return df[columns].copy()

    def create_combined_category(
        self,
        df: pd.DataFrame,
        col1: str,
        col2: str,
        new_col: str | None = None,
        sep: str = " / ",
    ) -> pd.DataFrame:
        out = df.copy()

        self._validate_columns(out, [col1, col2])

        new_col = new_col or f"{col1}_{col2}"

        out[new_col] = (
            out[col1].fillna("").astype(str)
            + sep
            + out[col2].fillna("").astype(str)
        )

        return out

    def create_date_bucket(
        self,
        df: pd.DataFrame,
        date_col: str,
        bucket: str,
        new_col: str | None = None,
    ) -> pd.DataFrame:
        out, generated_col = self._bucket_date(
            df,
            date_col,
            bucket,
        )

        if new_col and new_col != generated_col:
            out = out.rename(
                columns={generated_col: new_col}
            )

        return out

    def select_top_entities(
        self,
        df: pd.DataFrame,
        entity_col: str,
        metric_col: str,
        n: int,
        aggregate: str = "sum",
        ascending: bool = False,
        filters: list[dict] | None = None,
        keep_all_rows: bool = True,
    ) -> pd.DataFrame:
        
        self._validate_columns(df,[entity_col, metric_col])
        if aggregate not in self.ALLOWED_AGGS:
            raise ValueError(f"Unsupported aggregate: {aggregate}")

        if n <= 0:
            raise ValueError("n must be greater than zero")
        
        ranking_data = df.copy()

        if filters:
            ranking_data = self.filter_rows(
                ranking_data,
                filters,
            )

        ranking = (
            ranking_data
            .groupby(entity_col, dropna=False, as_index=False)[metric_col]
            .agg(aggregate)
            .sort_values(metric_col, ascending=ascending)
            .head(n)
        )

        selected_entities = ranking[entity_col].tolist()

        if keep_all_rows:
            return df[df[entity_col].isin(selected_entities)].copy()

        return ranking



    def run_plot_tool(
        self,
        tool_name: str,
        df: pd.DataFrame,
        args: dict[str, Any],
    ) -> dict:
        plotting_tools = {
            "plot_bar_chart": self.plot_bar_chart,
            "plot_line_chart": self.plot_line_chart,
            "plot_pie_chart": self.plot_pie_chart,
            "plot_scatter_chart": self.plot_scatter_chart,
            "plot_list": self.plot_list,
        }

        if tool_name not in plotting_tools:
            raise ValueError(f"Unknown plot tool: {tool_name}")

        args = self._filter_args(tool_name, args)
        return plotting_tools[tool_name](df=df, **args)

    def format_label(self, name: str) -> str:
        """
        Convert column-like names to display labels.

        Examples:
        - year_quarter -> Year quarter
        - percentage_contribution -> Percentage contribution
        - TOTAL_WATER_INJECTION_VOLUME -> Total water injection volume
        """
        if name is None:
            return ""

        text = str(name).replace("_", " ").strip().lower()
        return text[:1].upper() + text[1:]

    def _as_list(self, value):
        if value is None:
            return []
        return [value] if isinstance(value, str) else list(value)

    def _strip_markdown_json(self, text: str) -> str:
        """
        Remove markdown code fences from LLM JSON responses.

        Examples:
        ```json
        {...}
        ```

        ->
        {...}
        """

        text = text.strip()

        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

        return text.strip()

    def _validate_columns(
        self,
        df: pd.DataFrame,
        columns: list[str],
        label: str = "column",
    ):
        """
        Validate that all requested columns exist in the dataframe.
        """

        missing = [c for c in columns if c not in df.columns]

        if missing:
            raise ValueError(f"Missing {label}(s): {missing}")

    def _filter_args(
        self,
        tool_name: str,
        args: dict[str, Any],
    ) -> dict[str, Any]:
        """
        Remove unsupported arguments generated by the LLM.
        """

        allowed_args = {
            "plot_bar_chart": {
                "x",
                "y",
                "aggregate",
                "group_by",
                "color_by",
                "orientation",
                "barmode",
                "title",
                "template",
            },
            "plot_line_chart": {
                "x",
                "y",
                "aggregate",
                "group_by","series_by",
                "color_by",
                "date_bucket",
                "cumulative",
                "title",
                "template",
            },
            "plot_pie_chart": {
                "labels",
                "values",
                "aggregate",
                "group_by",
                "title",
                "hole",
                "template",
            },
            "plot_scatter_chart": {
                "x",
                "y",
                "color_by",
                "size_by",
                "text_by",
                "title",
                "template",
            },
            "plot_list": {
                "columns",
                "sort_by",
                "sort_order",
                "limit",
                "title",
            },
        }

        if tool_name not in allowed_args:
            raise ValueError(f"Unknown tool: {tool_name}")

        return {
            k: v
            for k, v in args.items()
            if k in allowed_args[tool_name]
        }

    def _aggregate(
        self,
        df: pd.DataFrame,
        group_by: list[str],
        value_cols: list[str],
        aggregate: str,
    ) -> pd.DataFrame:
        if aggregate not in self.ALLOWED_AGGS:
            raise ValueError(f"Unsupported aggregate: {aggregate}")

        self._validate_columns(df, group_by, "group_by column")
        self._validate_columns(df, value_cols, "value column")

        return (
            df.groupby(group_by, dropna=False, as_index=False)[value_cols]
            .agg(aggregate)
        )

    def _bucket_date(
        self,
        df: pd.DataFrame,
        date_col: str,
        bucket: str,
    ) -> tuple[pd.DataFrame, str]:
        """
        Create a date bucket column.

        bucket:
        - "D": day
        - "W": week
        - "M": month
        - "Q": quarter
        - "Y": year
        """

        out = df.copy()
        bucket_col = f"{date_col}_{bucket}"

        self._validate_columns(out, [date_col])

        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

        if bucket == "D":
            out[bucket_col] = out[date_col].dt.to_period("D").dt.to_timestamp()
        elif bucket == "W":
            out[bucket_col] = out[date_col].dt.to_period("W").dt.start_time
        elif bucket == "M":
            out[bucket_col] = out[date_col].dt.to_period("M").dt.to_timestamp()
        elif bucket == "Q":
            out[bucket_col] = out[date_col].dt.to_period("Q").dt.to_timestamp()
        elif bucket == "Y":
            out[bucket_col] = out[date_col].dt.to_period("Y").dt.to_timestamp()
        else:
            raise ValueError("date_bucket must be one of: D, W, M, Q, Y")

        return out, bucket_col

    def plot_bar_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        color_by: str | None = None,
        orientation: str = "v",
        barmode: str = "group",
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        required = [x, *y_cols]
        if color_by:
            required.append(color_by)

        self._validate_columns(df, required)

        work = df.copy()

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [x]

            if x not in group_cols:
                group_cols.insert(0, x)

            if color_by and color_by not in group_cols:
                group_cols.append(color_by)

            work = self._aggregate(work, group_cols, y_cols, aggregate)

        data = []
        groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

        for group_value, g in groups:
            for y_col in y_cols:
                if group_value is None:
                    name = self.format_label(y_col)
                else:
                    name = self.format_label(str(group_value))

                if group_value is not None and len(y_cols) > 1:
                    name = f"{self.format_label(str(group_value))} - {self.format_label(y_col)}"

                trace = {
                    "type": "bar",
                    "name": name,
                    "orientation": orientation,
                }

                if orientation == "h":
                    trace["x"] = g[y_col].tolist()
                    trace["y"] = g[x].astype(str).tolist()
                else:
                    trace["x"] = g[x].astype(str).tolist()
                    trace["y"] = g[y_col].tolist()

                data.append(trace)

        y_label = self.format_label(", ".join(y_cols))
        x_label = self.format_label(x)

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": title or f"{y_label} by {x_label}"
                },
                "xaxis": {
                    "title": {
                        "text": y_label if orientation == "h" else x_label
                    }
                },
                "yaxis": {
                    "title": {
                        "text": x_label if orientation == "h" else y_label
                    }
                },
                "barmode": barmode,
                # "template": template,
            },
            "config": {
                "responsive": True,
                "displaylogo": False,
            },
        }

    def plot_line_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        series_by: str | None = None,
        color_by: str | None = None,  # backwards compatibility
        date_bucket: str | None = None,
        cumulative: bool = False,
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        if series_by is None:
            series_by = color_by

        required = [x, *y_cols]
        if series_by:
            required.append(series_by)

        self._validate_columns(df, required)

        work = df.copy()
        x_plot = x

        if date_bucket is not None:
            work, x_plot = self._bucket_date(work, x, date_bucket)

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [x_plot]

            if x_plot not in group_cols:
                group_cols.insert(0, x_plot)

            if series_by and series_by not in group_cols:
                group_cols.append(series_by)

            work = self._aggregate(work, group_cols, y_cols, aggregate)

        sort_cols = [series_by, x_plot] if series_by else [x_plot]
        work = work.sort_values(sort_cols)

        if cumulative:
            if series_by:
                for col in y_cols:
                    work[col] = work.groupby(series_by, dropna=False)[col].cumsum()
            else:
                for col in y_cols:
                    work[col] = work[col].cumsum()

        data = []
        groups = work.groupby(series_by, dropna=False) if series_by else [(None, work)]

        total_points = len(work) * len(y_cols)
        disable_all_markers = total_points > 2000

        for group_value, g in groups:
            for y_col in y_cols:
                if group_value is None:
                    name = self.format_label(y_col)
                elif len(y_cols) == 1:
                    name = str(group_value)
                else:
                    name = f"{group_value} - {self.format_label(y_col)}"

                data.append({
                    "type": "scatter",
                    "mode": "lines",
                    "x": g[x_plot].tolist(),
                    "y": g[y_col].tolist(),
                    "name": name,
                })

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": self.format_label(title) or f"{', '.join(y_cols)} over {x}"
                },
                "xaxis": {
                    "title": {
                        "text": self.format_label(x)
                    }
                },
                "yaxis": {
                    "title": {
                        "text": self.format_label(", ".join(y_cols))
                    }
                },
            },
            "config": self.plotly_config,
        }

    def plot_pie_chart(
        self,
        df: pd.DataFrame,
        labels: str,
        values: str,
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        title: str | None = None,
        hole: float = 0.0,
        # template: str = "plotly_white",
    ) -> dict:
        self._validate_columns(df, [labels, values])

        work = df.copy()

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [labels]

            if labels not in group_cols:
                group_cols.insert(0, labels)

            work = self._aggregate(work, group_cols, [values], aggregate)

        return {
            "data": [
                {
                    "type": "pie",
                    "labels": work[labels].astype(str).tolist(),
                    "values": work[values].tolist(),
                    "hole": hole,
                }
            ],
            "layout": {
                "title": {
                    "text": title or f"{values} share by {labels}"
                },
                # "template": template,
            },
            "config": self.plotly_config
        }

    def plot_scatter_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        color_by: str | None = None,
        size_by: str | None = None,
        text_by: str | None = None,
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        required = [x, *y_cols]
        if color_by:
            required.append(color_by)
        if size_by:
            required.append(size_by)
        if text_by:
            required.append(text_by)

        self._validate_columns(df, required)

        work = df.copy()
        data = []
        groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

        total_points = len(work) * len(y_cols)
        disable_all_markers = total_points > 2000

        for group_value, g in groups:
            for y_col in y_cols:
                name = y_col if group_value is None else str(group_value)

                if group_value is not None and len(y_cols) > 1:
                    name = f"{group_value} - {y_col}"

                n_points = len(g)

                use_markers = (
                    not disable_all_markers
                    and n_points <= 100
                )

                mode = "markers" if use_markers else "lines"

                trace = {
                    "type": "scattergl",
                    "mode": mode,
                    "x": g[x].tolist(),
                    "y": g[y_col].tolist(),
                    "name": name,
                }

                if size_by:
                    size_values = pd.to_numeric(g[size_by], errors="coerce").fillna(0)
                    max_size = max(float(size_values.max()), 1.0)

                    trace["marker"] = {
                        "size": size_values.tolist(),
                        "sizemode": "area",
                        "sizeref": max_size / 40,
                        "sizemin": 4,
                    }

                if text_by:
                    trace["text"] = g[text_by].astype(str).tolist()
                    trace["hovertemplate"] = (
                        f"{x}: %{{x}}<br>"
                        f"{y_col}: %{{y}}<br>"
                        f"{text_by}: %{{text}}"
                        "<extra></extra>"
                    )

                data.append(trace)

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": title or f"{', '.join(y_cols)} vs {x}"
                },
                "xaxis": {
                    "title": {
                        "text": x
                    }
                },
                "yaxis": {
                    "title": {
                        "text": ", ".join(y_cols)
                    }
                },
                # "template": template,
            },
            "config": self.plotly_config
        }

    def plot_list(
        self,
        df: pd.DataFrame,
        columns: list[str] | str | None = None,
        *,
        sort_by: str | None = None,
        sort_order: str = "desc",
        limit: int | None = None,
        title: str | None = None,
    ) -> dict:
        work = df.copy()

        if columns is not None:
            columns = self._as_list(columns)
            self._validate_columns(work, columns)
            work = work[columns]

        if sort_by is not None:
            self._validate_columns(work, [sort_by])

            ascending = sort_order.lower() == "asc"
            work = work.sort_values(sort_by, ascending=ascending)

        if limit is not None:
            work = work.head(limit)

        header_values = [self.format_label(c) for c in work.columns]

        cell_values = []
        for col in work.columns:
            s = work[col]

            if pd.api.types.is_datetime64_any_dtype(s):
                values = s.dt.strftime("%Y-%m-%d").fillna("").tolist()
            else:
                values = s.fillna("").astype(str).tolist()

            cell_values.append(values)

        return {
            "data": [
                {
                    "type": "table",
                    "header": {
                        "values": header_values,
                        "align": "left",
                    },
                    "cells": {
                        "values": cell_values,
                        "align": "left",
                    },
                }
            ],
            "layout": {
                "title": {
                    "text": self.format_label(title) or "Table"
                },
            },
            "config": self.plotly_config
        }
    

class PresenterConfig:

    prompt :str = chart_agent_prompt
    split_subinstructions_prompr: str = split_subinstructions_prompt 
    small_table_prompt: str = small_table_prompt 
    
    def __init__(
        self,
        charting_tools: PresenterChartingTools | None = None,
    ):
        self.charting_tools = charting_tools or PresenterChartingTools()


class PresenterComponent4:
    """
    Converts an ExecutorState into UI display items.

    Each TaskResult is processed as a whole:
    - all TextResult and DataFrameResult objects are added to one context;
    - one LLM call splits the task instruction into sub-instructions;
    - each sub-instruction is associated with one source;
    - text sources become text UIItems;
    - table sources are passed to the existing dataframe presentation logic.
    """

    def __init__(
        self,
        llm: Any,
        config: PresenterConfig | None = None,
    ):
        self.llm = llm
        self.config = config or PresenterConfig()
        self.charting_tools = self.config.charting_tools

    def run(self, result_state: ExecutorState) -> PresenterResponse:
        return self.process_task_results(result_state)

    def _make_clarification_item(
        self,
        clarification_request: str,
    ) -> UIItem:
        return UIItem(
            id=f"question_{uuid4().hex[:8]}",
            type="question",
            title="Additional information required",
            data={"question": clarification_request},
        )

    def _make_text_item(
        self,
        data_result: TextResult,
    ) -> UIItem:
        return UIItem(
            id=f"text_{uuid4().hex[:8]}",
            type="text",
            title=None,
            data={"text": data_result.text},
        )

    def _make_error_item(
        self,
        task_result: TaskResult,
        data_result: object | None = None,
    ) -> UIItem:
        return UIItem(
            id=f"error_{uuid4().hex[:8]}",
            type="error",
            title="Presentation error",
            data={
                "message": f"No presenter for result from {task_result.agent}",
                "details": (
                    str(type(data_result))
                    if data_result is not None
                    else task_result.instruction
                ),
            },
        )

    def build_task_result_context(
        self,
        task_result: TaskResult,
    ) -> tuple[str, dict[str, TextResult | DataFrameResult]]:
        """
        Build:
        - one text context containing all TextResult and DataFrameResult objects;
        - a source map used later to recover the original result objects.
        """
        processor = TableResponseProcessor()

        context_parts: list[str] = []
        source_map: dict[str, TextResult | DataFrameResult] = {}

        for n, data_result in enumerate(task_result.data_results):

            if isinstance(data_result, TextResult):
                source_id = f"text_{n}"

                context_parts.append(
                    "\n".join([
                        f"SOURCE_ID: {source_id}",
                        "SOURCE_TYPE: text",
                        "CONTENT:",
                        data_result.text,
                    ])
                )

                source_map[source_id] = data_result

            elif isinstance(data_result, DataFrameResult):
                source_id = data_result.table_name
                table_context = processor.extract_table_context(data_result)

                context_parts.append(
                    "\n".join([
                        f"SOURCE_ID: {source_id}",
                        "SOURCE_TYPE: table",
                        table_context,
                    ])
                )

                source_map[source_id] = data_result

        context_text = "\n\n---\n\n".join(context_parts)

        return context_text, source_map

    def get_subinstructions(
        self,
        task_result: TaskResult,
        context_text: str,
    ) -> SubInstructions:
        """
        Split the task instruction and associate each sub-instruction
        with one available source.
        """
        messages = [
            SystemMessage(content=self.config.split_subinstructions_prompr),
            HumanMessage(
                content=(
                    f"INSTRUCTION\n"
                    f"{task_result.instruction}\n\n"
                    f"AVAILABLE SOURCES\n"
                    f"{context_text}"
                )
            ),
        ]

        structured_llm = self.llm.with_structured_output(SubInstructions)

        return structured_llm.invoke(messages)

    def process_single_task_result(
        self,
        task_result: TaskResult,
    ) -> list[UIItem]:
        context_text, source_map = self.build_task_result_context(
            task_result
        )

        sub_instructions = self.get_subinstructions(
            task_result=task_result,
            context_text=context_text,
        )

        ui_items: list[UIItem] = []

        for item in sub_instructions.items:
            source = source_map[item.source_id]

            if item.kind == "text":
                ui_items.append(
                    self._make_text_item(source)
                )

            elif item.kind == "table":
                ui_items.append(
                    self._process_dataframe(
                        source,
                        item.sub_instruction,
                    )
                )

        return ui_items

    def process_task_results(
        self,
        execution_state: ExecutorState,
    ) -> PresenterResponse:
        ui_items: list[UIItem] = []

        clarification_request = execution_state.get(
            "clarification_request"
        )

        if clarification_request:
            ui_items.append(
                self._make_clarification_item(
                    clarification_request
                )
            )

            return PresenterResponse(items=ui_items)

        for task_result in execution_state.get("task_results", []):
            ui_task_items = self.process_single_task_result(
                task_result
            )

            ui_items.extend(ui_task_items)

        return PresenterResponse(items=ui_items)

    def _present_very_small_table(
        self,
        df: pd.DataFrame,
        data_result: DataFrameResult,
        instruction: str,
    ) -> UIItem:
        data_string = df.to_json()

        print('processing very small table')
        prompt = (
            self.config.small_table_prompt
            + "\n\n"
            + (
                "### Context\n"
                f"- Table Name: {getattr(data_result, 'table_name', 'N/A')}\n"
                f"- Description: "
                f"{getattr(data_result, 'description', 'No description provided.')}\n\n"
                "### Data\n"
                f"{data_string}\n\n"
                "User question:\n"
                f"{instruction}\n"
            )
        )

        response = self.llm.invoke(prompt)

        text_output = (
            response.content
            if hasattr(response, "content")
            else str(response)
        )

        return UIItem(
            id=f"text_{uuid4().hex[:8]}",
            type="text",
            title=None,
            data={"text": text_output},
        )

    def _make_chart_item(
        self,
        figure_title: str,
        plotly_json_figure: dict,
        description: str | None = None,
    ) -> UIItem:
        return UIItem(
            id=f"chart_{uuid4().hex[:8]}",
            type="chart",
            title=figure_title,
            data={
                "engine": "plotly",
                "plotly": plotly_json_figure,
            },
            meta={
                "description": description,
            },
        )


    def _run_chart_plan(
        self,
        plan: dict,
        df: pd.DataFrame,
    ) -> dict | None:
        work = df.copy()

        preprocess_steps = plan.get("preprocess") or []
        plot = plan.get("plot")

        if preprocess_steps:
            work = self.charting_tools.run_preprocess(
                work,
                preprocess_steps,
            )

        if not plot:
            return None

        tool_name = plot.get("tool")
        args = plot.get("args") or {}

        return self.charting_tools.run_plot_tool(
            tool_name=tool_name,
            df=work,
            args=args,
        )


    def _format_label(
        self,
        name: str,
    ) -> str:
        if name is None:
            return ""

        text = str(name).replace("_", " ").strip().lower()

        return text[:1].upper() + text[1:]

    def _process_dataframe(
        self,
        data_result: DataFrameResult,
        instruction: str,
    ) -> UIItem:
        df = data_result.dataframe
        nrows, ncols = df.shape

        if nrows <= 2 and ncols <= 2:
            return self._present_very_small_table(
                df,
                data_result,
                instruction,
            )

        processor = TableResponseProcessor()

        table_context = processor.extract_table_context(
            data_result
        )

        chart_plan = self._select_chart_plan(
            instruction,
            table_context,
        )

        print('****chart plan****')
        print(instruction)
        print(chart_plan)


        chart_output = self._run_chart_plan(
            chart_plan,
            df,
        )

        return self._make_chart_item(
            self._format_label(data_result.table_name),
            chart_output,
            data_result.description,
        )

    def _strip_markdown_json(
        self,
        text: str,
    ) -> str:
        text = text.strip()

        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
        )

        text = re.sub(
            r"\s*```$",
            "",
            text,
        )

        return text.strip()

    def _select_chart_plan(
        self,
        user_query: str,
        table_context: str,
    ) -> dict:
        messages = [
            SystemMessage(content=self.config.prompt),
            HumanMessage(
                content=(
                    f"USER QUERY\n"
                    f"{user_query}\n\n"
                    f"TABLE\n"
                    f"{table_context}"
                )
            ),
        ]

        response = self.llm.invoke(messages)

        text = self._strip_markdown_json(
            response.content
        )

        return json.loads(text)
    

    

In [ ]:
#pprint.pprint( execution_state['task_results'][1] )
t = execution_state['task_results'][1]
instruction = t.instruction 
#t.data_results.insert(0, TextResult(text="All the fruits in the basket are sweet."))

print( instruction )
p = TableResponseProcessor()
context = [] 

aux= {} 
for n,data_result in enumerate(t.data_results):


    if isinstance( data_result, TextResult):
        print("processing text ")
        context.append( data_result.text )
        aux[n] = data_result.text 

    if isinstance(data_result, DataFrameResult):
        print("processing dataframe result")
        table_context = p.extract_table_context( data_result )
        context.append( table_context )
        aux[ data_result.table_name ] = (table_context,data_result)
    
prompt = """You will receive an 'instruction' and information of tables and text. your job is to 
analyze the instruction. It might contain several sub-instructions. 
Decide what parts of the information available can be used to execute the 
instruction and its sub-instructions. 

Do not explain anything, do not add more details than strictly needed to produce the required output 
"""
class SubInstruction(BaseModel):
    sub_instruction: str = Field(
        description="The specific part of the instruction."
    )
    kind:  Literal['table','text']
    
    data: str  = Field(
        description="The name of the table or the textual information "
    )
  
  
class SubInstructions(BaseModel):

    items: List[SubInstruction] =  Field(description="The list of sub-instructions and the information relevant for each")

context = "\n\n".join(context)

#instruction2 = "Check if the fruits are sour or sweet, " + instruction# then list the top 5 producers by cummulated oil produced in 2018 "

context =  "\n" + context + "\n\n" + instruction + "\n\n"
messages = [SystemMessage(prompt + "\n\n" + context )]

structured_llm = llm.with_structured_output(SubInstructions)
response = structured_llm.invoke(messages)

In [ ]:
pprint.pprint(context)


In [ ]:
pprint.pprint(response.items)


In [ ]:

items = [] 

for i in response.items:
    if i.kind=='text':
        print("It is a text")
        r =  UIItem(
                id=f"text_{uuid4().hex[:8]}",
                type="text",
                title=None,
                data={"text": i.data}
            )
        items.append( r )

    if i.kind=='table':

        #print( '**',i.sub_instruction,'**')
        table_name = i.data 
        acontext = aux[table_name ][0]
        data_result = aux[table_name][1]
        #print('table', table_name, acontext )
        
        r = presenter._process_dataframe(data_result,i.sub_instruction)
        items.append( r )

        


In [ ]:
print( items )

print()
print()
print()

for item in items:

    if item.type=='text':
        print( item.data['text'])

    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


## More organized

In [ ]:
from typing import Literal
from uuid import uuid4

from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage


# =============================================================================
# Structured output models
# =============================================================================

class SubInstruction(BaseModel):
    sub_instruction: str = Field(
        description=(
            "The specific part of the original instruction that must be "
            "presented using the selected source."
        )
    )

    kind: Literal["table", "text"] = Field(
        description="The type of source associated with this sub-instruction."
    )

    source_id: str = Field(
        description=(
            "The exact SOURCE ID provided in the available sources. "
            "For a table, this is the exact table name. "
            "For text, this is the exact text result identifier."
        )
    )


class SubInstructions(BaseModel):
    items: list[SubInstruction] = Field(
        description=(
            "The requested outputs, in the order in which they should be "
            "presented. Irrelevant and intermediate sources must be omitted."
        )
    )


# =============================================================================
# Routing prompt
# =============================================================================

TASK_PRESENTATION_ROUTER_PROMPT = """
You receive one instruction and a set of available sources.

The instruction may contain several sub-instructions.

Your job is to:

1. Identify each distinct output explicitly requested by the instruction.
2. Match each requested output to exactly one relevant source.
3. Return the outputs in the order in which they should be presented.
4. Use the exact SOURCE ID provided for each source.
5. Omit sources that are irrelevant or only intermediate calculation results.
6. Do not invent facts, tables, source IDs, calculations, or additional requests.
7. Do not explain your decisions.
8. Do not create an item when the available sources cannot support it.
9. Do not repeat the same source unless it is genuinely required for two
   different requested outputs.

For a text source:
- Use it when the source directly contains the requested textual answer.
- The sub_instruction should describe the part of the instruction answered
  by the text.
- The source_id must be the exact text SOURCE ID.

For a table source:
- Use it when the table contains the information required for the requested
  table, chart, list, ranking, comparison, or numerical presentation.
- The sub_instruction must contain only the part of the original instruction
  that the selected table can address.
- The source_id must be the exact table SOURCE ID.

Important:
- A table used only to calculate another final table is usually an intermediate
  source and should be omitted unless the user explicitly requested it.
- Do not return the source content itself.
- Return only the structured result.
"""


# =============================================================================
# Select the TaskResult to process
# =============================================================================

task_result_index = 1

task_result = execution_state["task_results"][task_result_index]
instruction = task_result.instruction

print("TASK INSTRUCTION")
print(instruction)
print()


# =============================================================================
# Build source context and source lookup
# =============================================================================

table_processor = TableResponseProcessor()

source_contexts: list[str] = []
source_lookup: dict[str, TextResult | DataFrameResult] = {}

for result_index, data_result in enumerate(task_result.data_results):

    if isinstance(data_result, TextResult):
        source_id = f"text_result_{result_index}"

        source_contexts.append(
            "\n".join([
                f"SOURCE ID: {source_id}",
                "SOURCE KIND: text",
                "CONTENT:",
                data_result.text,
            ])
        )

        source_lookup[source_id] = data_result

    elif isinstance(data_result, DataFrameResult):
        source_id = data_result.table_name
        table_context = table_processor.extract_table_context(data_result)

        source_contexts.append(
            "\n".join([
                f"SOURCE ID: {source_id}",
                "SOURCE KIND: table",
                table_context,
            ])
        )

        source_lookup[source_id] = data_result

    else:
        print(
            "Ignoring unsupported data result:",
            type(data_result).__name__,
        )


available_sources_context = "\n\n---\n\n".join(source_contexts)

print("AVAILABLE SOURCE IDS")
for source_id, source in source_lookup.items():
    print(f"- {source_id}: {type(source).__name__}")
print()


# =============================================================================
# One LLM call to split the instruction and route each part to a source
# =============================================================================

messages = [
    SystemMessage(content=TASK_PRESENTATION_ROUTER_PROMPT),
    HumanMessage(
        content=(
            f"ORIGINAL INSTRUCTION\n"
            f"{instruction}\n\n"
            f"AVAILABLE SOURCES\n"
            f"{available_sources_context}"
        )
    ),
]

structured_llm = llm.with_structured_output(SubInstructions)
routing_response = structured_llm.invoke(messages)

print("ROUTING RESPONSE")
for routed_item in routing_response.items:
    print(routed_item)
print()


# =============================================================================
# Convert the routed outputs into UIItems
# =============================================================================

items: list[UIItem] = []

for routed_item in routing_response.items:
    source = source_lookup.get(routed_item.source_id)

    if source is None:
        items.append(
            UIItem(
                id=f"error_{uuid4().hex[:8]}",
                type="error",
                title="Presentation error",
                data={
                    "message": (
                        "The presentation router selected an unknown source."
                    ),
                    "details": routed_item.source_id,
                },
            )
        )
        continue

    if routed_item.kind == "text":
        if not isinstance(source, TextResult):
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            "The presentation router classified a non-text "
                            "source as text."
                        ),
                        "details": routed_item.source_id,
                    },
                )
            )
            continue

        items.append(
            UIItem(
                id=f"text_{uuid4().hex[:8]}",
                type="text",
                title=None,
                data={
                    "text": source.text,
                },
                meta={
                    "sub_instruction": routed_item.sub_instruction,
                    "source_id": routed_item.source_id,
                },
            )
        )

    elif routed_item.kind == "table":
        if not isinstance(source, DataFrameResult):
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            "The presentation router classified a non-table "
                            "source as a table."
                        ),
                        "details": routed_item.source_id,
                    },
                )
            )
            continue

        try:
            ui_item = presenter._process_dataframe(
                source,
                routed_item.sub_instruction,
            )

            if ui_item is not None:
                ui_item.meta = {
                    **ui_item.meta,
                    "sub_instruction": routed_item.sub_instruction,
                    "source_id": routed_item.source_id,
                }
                items.append(ui_item)

        except Exception as exc:
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            f"Could not present table "
                            f"{routed_item.source_id}."
                        ),
                        "details": str(exc),
                    },
                    meta={
                        "sub_instruction": routed_item.sub_instruction,
                        "source_id": routed_item.source_id,
                    },
                )
            )


# =============================================================================
# Final presenter response
# =============================================================================

presenter_response = PresenterResponse(items=items)

print("GENERATED UI ITEMS")
for item in presenter_response.items:
    print(
        {
            "id": item.id,
            "type": item.type,
            "title": item.title,
            "source_id": item.meta.get("source_id"),
            "sub_instruction": item.meta.get("sub_instruction"),
        }
    )

presenter_response
  

In [ ]:
items = presenter_response.items 
print( items )

print()
print()
print()

for item in items:

    if item.type=='text':
        print( item.data['text'])

    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


In [ ]:


presenter = PresenterComponent4( llm )
#presenter = PresenterComponent2( llm )

ui_items = presenter.run( execution_state )
ui_items

In [ ]:
ui_items.items

In [ ]:
item = ui_items.items[1]
item = item.data['plotly']
pio.show(item)

item = ui_items.items[2]
item = item.data['plotly']
pio.show(item)

item = ui_items.items[3]
item = item.data['plotly']
pio.show(item)